# 01 - Extract the Original 224x224 Dataset

This notebook has two jobs: extract the single downloaded `KneeXrayData.zip` archive and export the full bilateral HDF5 radiographs as PNG files.

It does not resize, crop, augment, relabel, or train a model. The original 224x224 PNG files remain in their published folder. The full bilateral PNG files are exported separately because one full image contains two knees and can have two different KL grades.


In [8]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
from zipfile import ZipFile

# One archive downloaded previously from Mendeley Data.
ARCHIVE = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/KneeXrayData.zip")
EXTRACT_DIR = Path("/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted")
FULL_DATASET = EXTRACT_DIR / "KneeXrayData"
ORIGINAL_224 = FULL_DATASET / "ClsKLData/kneeKL224"
H5_ROOT = FULL_DATASET / "DetKneeData/H5"

if not ARCHIVE.is_file():
    raise FileNotFoundError(f"ZIP file not found: {ARCHIVE}")

# Both sources are required: 224x224 classifier images for notebook 02 and
# full bilateral HDF5 radiographs for notebook 03.
if not ORIGINAL_224.is_dir() or not H5_ROOT.is_dir():
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with ZipFile(ARCHIVE) as zip_file:
        zip_file.extractall(EXTRACT_DIR)

for required in (ORIGINAL_224, H5_ROOT):
    if not required.is_dir():
        raise FileNotFoundError(f"Required folder not found after extraction: {required}")

print("Full extracted dataset:", FULL_DATASET)
print("Original 224x224 classifier dataset:", ORIGINAL_224)
print("Full bilateral HDF5 images:", H5_ROOT)
print("Notebook 02 uses train/ and val/ inside this folder.")
print("Notebook 03 uses the full extracted HDF5 images inside this archive.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full extracted dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData
Original 224x224 classifier dataset: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/ClsKLData/kneeKL224
Full bilateral HDF5 images: /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/extracted/KneeXrayData/DetKneeData/H5
Notebook 02 uses train/ and val/ inside this folder.
Notebook 03 uses the full extracted HDF5 images inside this archive.


## Export full bilateral HDF5 images as PNG files

The output layout is `full_bilateral_png/train`, `full_bilateral_png/val`, and `full_bilateral_png/test`. A full bilateral image is stored once by patient ID, not inside a grade folder, because the right and left knees may have different grades.


In [9]:
import cv2
import h5py
import numpy as np

FULL_PNG_ROOT = Path(
    "/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/"
    "full_bilateral_png_v1"
)

for split in ("train", "val", "test"):
    h5_dir = H5_ROOT / f"{split}H5"
    if not h5_dir.is_dir():
        raise FileNotFoundError(f"HDF5 folder not found: {h5_dir}")

    output_dir = FULL_PNG_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    h5_paths = sorted(h5_dir.glob("*.h5"))

    for index, h5_path in enumerate(h5_paths, start=1):
        output_path = output_dir / f"{h5_path.stem}.png"
        if output_path.is_file():
            continue  # Keep completed PNG files when a Colab session reconnects.

        with h5py.File(h5_path, "r") as h5_file:
            image = np.asarray(h5_file["images"])
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2)
        if image.shape[-1] == 1:
            image = np.repeat(image, 3, axis=2)
        image = np.clip(image[..., :3], 0, 255).astype(np.uint8)
        image_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        if not cv2.imwrite(str(output_path), image_bgr):
            raise RuntimeError(f"Cannot write PNG: {output_path}")

        if index % 100 == 0 or index == len(h5_paths):
            print(f"{split}: {index}/{len(h5_paths)} full PNG images")

print("Full bilateral PNG folder:", FULL_PNG_ROOT)


train: 100/2889 full PNG images
train: 200/2889 full PNG images
train: 300/2889 full PNG images
train: 400/2889 full PNG images
train: 500/2889 full PNG images
train: 600/2889 full PNG images
train: 700/2889 full PNG images
train: 800/2889 full PNG images
train: 900/2889 full PNG images
train: 1000/2889 full PNG images
train: 1100/2889 full PNG images
train: 1200/2889 full PNG images
train: 1300/2889 full PNG images
train: 1400/2889 full PNG images
train: 1500/2889 full PNG images
train: 1600/2889 full PNG images
train: 1700/2889 full PNG images
train: 1800/2889 full PNG images
train: 1900/2889 full PNG images
train: 2000/2889 full PNG images
train: 2100/2889 full PNG images
train: 2200/2889 full PNG images
train: 2300/2889 full PNG images
train: 2400/2889 full PNG images
train: 2500/2889 full PNG images
train: 2600/2889 full PNG images
train: 2700/2889 full PNG images
train: 2800/2889 full PNG images
train: 2889/2889 full PNG images
val: 100/413 full PNG images
val: 200/413 full PNG i